# Imports

In [1]:
from pathlib import Path
import timeit
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, recall_score

# File Paths


In [2]:
KERAS_DIR = Path("../models/keras")
TFLITE_DIR = Path("../models/tflite")
DATA_PATH = Path("../data/fdia_dataset_processed.npz")

print("Keras:", KERAS_DIR.resolve())
print("TFLite:", TFLITE_DIR.resolve())
print("Dataset:", DATA_PATH.resolve())

Keras: C:\Users\nalui\TinyML_Research_Inference\models\keras
TFLite: C:\Users\nalui\TinyML_Research_Inference\models\tflite
Dataset: C:\Users\nalui\TinyML_Research_Inference\data\fdia_dataset_processed.npz


# Dataset

In [3]:
data = np.load(DATA_PATH)

X_test = data["X_test"].astype(np.float32)
y_test = data["y_test"]

X_test_temporal = np.transpose(X_test, (0, 2, 1))

print("MLP input:", X_test.shape)
print("CNN/LSTM input:", X_test_temporal.shape)
print("Labels:", y_test.shape)

MLP input: (9720, 6, 83)
CNN/LSTM input: (9720, 83, 6)
Labels: (9720,)


## Subsample

In [34]:
TIMING_FRACTION = 0.1029
RUNS = 100
SEED = 42

np.random.seed(SEED)

sample_size = int(TIMING_FRACTION * len(X_test))
timing_idx = np.random.choice(len(X_test), size=sample_size, replace=False)

print("Timing fraction:", TIMING_FRACTION)
print("Timing samples:", len(timing_idx))
print("Runs:", RUNS)

Timing fraction: 0.1029
Timing samples: 1000
Runs: 100


# Functions

In [ ]:
def prepare_inputs(X, input_details):
    X = X.astype(np.float32)
    dtype = input_details["dtype"]

    if dtype != np.float32:
        scale, zero_point = input_details["quantization"]
        limits = np.iinfo(dtype)
        X = np.clip(np.round(X / scale + zero_point), limits.min, limits.max).astype(dtype)

    return np.expand_dims(X, axis=1)


def predict_tflite(model_path, X):
    interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=1 )
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    X_ready = prepare_inputs(X, input_details)

    predictions = np.empty(len(X), dtype=np.float32)

    for i, x in enumerate(X_ready):
        interpreter.set_tensor(input_details["index"], x)
        interpreter.invoke()

        output = interpreter.get_tensor(output_details["index"]).squeeze()

        if output_details["dtype"] != np.float32:
            scale, zero_point = output_details["quantization"]
            output = (float(output) - zero_point) * scale

        predictions[i] = output

    return predictions


def measure_inference_time_tflite(model_path, X_timing, runs=100):
    interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=1)

    input_details = interpreter.get_input_details()[0]
    input_index = input_details["index"]

    interpreter.resize_tensor_input(input_index, [len(X_timing), *X_timing.shape[1:]], strict=False)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    X_ready = X_timing.astype(np.float32)

    if input_details["dtype"] != np.float32:
        scale, zero_point = input_details["quantization"]
        limits = np.iinfo(input_details["dtype"])
        X_ready = np.clip(np.round(X_ready / scale + zero_point), limits.min, limits.max).astype(input_details["dtype"])

    # Set input once
    interpreter.set_tensor(input_index, X_ready)

    # Warm-up
    interpreter.invoke()

    run_times = []

    for _ in range(runs):
        start = timeit.default_timer()

        interpreter.invoke()

        end = timeit.default_timer()

        run_times.append(((end - start) * 1000) / len(X_timing))

    return np.mean(run_times)


def evaluate_tflite(model_path, keras_path, model_name, architecture):
    X_model = X_test_temporal if architecture in ["CNN1D", "LSTM"] else X_test
    X_timing = X_model[timing_idx]

    print(f"\nTesting: {model_name}")

    predictions = predict_tflite(model_path, X_model)
    y_pred = (predictions >= 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    fdia_recall = recall_score(y_test, y_pred, pos_label=1)
    fault_recall = recall_score(y_test, y_pred, pos_label=0)

    keras_model = tf.keras.models.load_model(keras_path, compile=False)
    parameters = keras_model.count_params()

    model_size_kb = model_path.stat().st_size / 1024
    inference_time = measure_inference_time_tflite(model_path, X_timing, RUNS)

    result = {
        "architecture": architecture,
        "model": model_name,
        "parameters": parameters,
        "accuracy": accuracy,
        "fdia_recall": fdia_recall,
        "fault_recall": fault_recall,
        "model_size_kb": model_size_kb,
        "inference_time_ms": inference_time
    }

    print(f"Parameters:     {parameters:,}")
    print(f"Accuracy:       {accuracy:.8f}")
    print(f"FDIA Recall:    {fdia_recall:.8f}")
    print(f"Fault Recall:   {fault_recall:.8f}")
    print(f"TFLite Size:    {model_size_kb:.6f} KB")
    print(f"Inference Time: {inference_time:.6f} ms/sample")

    return result

results = []

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3224218874.py, line 39)

# CNN Models tflite

In [6]:
cnn_tflite_models = [
    ("CNN1D_Original_baseline", "CNN1D_Original_baseline.tflite", "CNN1D_HPO.keras"),
    ("CNN1D_Original_quant", "CNN1D_Original_quant.tflite", "CNN1D_HPO.keras"),

    ("CNN1D_Node_baseline", "CNN1D_Node_baseline.tflite", "CNN1D_NodePruned.keras"),
    ("CNN1D_Node_quant", "CNN1D_Node_quant.tflite", "CNN1D_NodePruned.keras"),

    ("CNN1D_Weight_baseline", "CNN1D_Weight_baseline.tflite", "CNN1D_WeightPruned.keras"),
    ("CNN1D_Weight_quant", "CNN1D_Weight_quant.tflite", "CNN1D_WeightPruned.keras"),

    ("CNN1D_WeightNode_baseline", "CNN1D_WeightNode_baseline.tflite", "CNN1D_WeightNodePruned.keras"),
    ("CNN1D_WeightNode_quant", "CNN1D_WeightNode_quant.tflite", "CNN1D_WeightNodePruned.keras"),

    ("CNN1D_NodeWeight_baseline", "CNN1D_NodeWeight_baseline.tflite", "CNN1D_NodeWeightPruned.keras"),
    ("CNN1D_NodeWeight_quant", "CNN1D_NodeWeight_quant.tflite", "CNN1D_NodeWeightPruned.keras")
]

## Run tests CNN tflite

In [7]:
for name, tflite_file, keras_file in cnn_tflite_models:
    result = evaluate_tflite(
        TFLITE_DIR / tflite_file,
        KERAS_DIR / keras_file,
        name,
        "CNN1D"
    )

    results.append(result)


Testing: CNN1D_Original_baseline


c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Parameters:     92,289
Accuracy:       0.99835391
FDIA Recall:    0.99934896
Fault Recall:   0.99745696
TFLite Size:    367.722656 KB
Inference Time: 0.037456 ms/sample

Testing: CNN1D_Original_quant


c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


...

Parameters:     48,751
Accuracy:       0.98950617
FDIA Recall:    0.99956597
Fault Recall:   0.98043818
TFLite Size:    60.804688 KB
Inference Time: 0.014871 ms/sample

Testing: CNN1D_NodeWeight_baseline


c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Parameters:     46,596
Accuracy:       0.99876543
FDIA Recall:    0.99934896
Fault Recall:   0.99823944
TFLite Size:    189.328125 KB
Inference Time: 0.021948 ms/sample

Testing: CNN1D_NodeWeight_quant


c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Parameters:     46,596
Accuracy:       0.99783951
FDIA Recall:    0.99869792
Fault Recall:   0.99706573
TFLite Size:    58.648438 KB
Inference Time: 0.014835 ms/sample


# MLP Models tflite

In [6]:
## MLP Models tflite
mlp_tflite_models = [
    ("MLP_Original_baseline", "MLP_Original_baseline.tflite", "MLP_HPO.keras"),
    ("MLP_Original_quant", "MLP_Original_quant.tflite", "MLP_HPO.keras"),

    ("MLP_Node_baseline", "MLP_Node_baseline.tflite", "MLP_NodePruned.keras"),
    ("MLP_Node_quant", "MLP_Node_quant.tflite", "MLP_NodePruned.keras"),

    ("MLP_Weight_baseline", "MLP_Weight_baseline.tflite", "MLP_WeightPruned.keras"),
    ("MLP_Weight_quant", "MLP_Weight_quant.tflite", "MLP_WeightPruned.keras"),

    ("MLP_WeightNode_baseline", "MLP_WeightNode_baseline.tflite", "MLP_WeightNodePruned.keras"),
    ("MLP_WeightNode_quant", "MLP_WeightNode_quant.tflite", "MLP_WeightNodePruned.keras"),

    ("MLP_NodeWeight_baseline", "MLP_NodeWeight_baseline.tflite", "MLP_NodeWeightPruned.keras"),
    ("MLP_NodeWeight_quant", "MLP_NodeWeight_quant.tflite", "MLP_NodeWeightPruned.keras")
]

## MLP Test

In [38]:
import pathlib
import numpy as np
import tensorflow as tf
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

KERAS_DIR = pathlib.Path("../Models/keras")
TFLITE_DIR = pathlib.Path("../Models/tflite")

models = {
    "Original": "MLP_HPO.keras",
    "Node": "MLP_NodePruned.keras",
    "Weight": "MLP_WeightPruned.keras",
    "WeightNode": "MLP_WeightNodePruned.keras",
    "NodeWeight": "MLP_NodeWeightPruned.keras"
}

data = np.load("../data/fdia_dataset_processed.npz")
X_train = data["X_train"].astype(np.float32)   # MLP: sem transpose
X_test = data["X_test"].astype(np.float32)
y_test = data["y_test"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

def representative_data_gen():
    for sample in X_train[:100]:
        yield [sample[np.newaxis, ...]]

def build_converter(model):
    input_shape = model.inputs[0].shape[1:]
    concrete_func = tf.function(model).get_concrete_function(
        tf.TensorSpec([None, *input_shape], model.inputs[0].dtype)
    )
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    return tf.lite.TFLiteConverter.from_concrete_functions([frozen_func], model)

for name, filename in models.items():
    print(f"\nConverting: {name}")
    model = tf.keras.models.load_model(KERAS_DIR / filename, compile=False)

    # Float TFLite
    converter = build_converter(model)
    tflite_model = converter.convert()
    baseline_path = TFLITE_DIR / f"MLP_{name}_baseline.tflite"
    baseline_path.write_bytes(tflite_model)

    # Full Integer PTQ
    converter = build_converter(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    tflite_model_quant = converter.convert()
    quant_path = TFLITE_DIR / f"MLP_{name}_quant.tflite"
    quant_path.write_bytes(tflite_model_quant)

    print(f"Baseline: {baseline_path.stat().st_size / 1024:.2f} KB")
    print(f"Quantized: {quant_path.stat().st_size / 1024:.2f} KB")

for name in models:
    path = TFLITE_DIR / f"MLP_{name}_quant.tflite"
    interpreter = tf.lite.Interpreter(model_path=str(path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    print(f"\n{name}")
    print("Shape signature:", input_details["shape_signature"])

X_train: (38880, 6, 83)
X_test: (9720, 6, 83)

Converting: Original


KeyboardInterrupt: 

In [11]:
## Run tests MLP tflite
for name, tflite_file, keras_file in mlp_tflite_models:
    result = evaluate_tflite(
        TFLITE_DIR / tflite_file,
        KERAS_DIR / keras_file,
        name,
        "MLP"
    )

    results.append(result)


Testing: MLP_Original_baseline


c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Parameters:     44,753
Accuracy:       0.99835391
FDIA Recall:    0.99826389
Fault Recall:   0.99843505
TFLite Size:    179.789062 KB
Inference Time: 0.000253 ms/sample

Testing: MLP_Original_quant


c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Parameters:     44,753
Accuracy:       0.99763374
FDIA Recall:    0.99739583
Fault Recall:   0.99784820
TFLite Size:    58.093750 KB
Inference Time: 0.001296 ms/sample

Testing: MLP_Node_baseline


...

Parameters:     28,206
Accuracy:       0.98765432
FDIA Recall:    0.97547743
Fault Recall:   0.99863067
TFLite Size:    39.960938 KB
Inference Time: 0.001322 ms/sample

Testing: MLP_NodeWeight_baseline


c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Parameters:     28,920
Accuracy:       0.99773663
FDIA Recall:    0.99761285
Fault Recall:   0.99784820
TFLite Size:    117.773438 KB
Inference Time: 0.000805 ms/sample

Testing: MLP_NodeWeight_quant


c:\Users\nalui\tensorflow_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Parameters:     28,920
Accuracy:       0.99825103
FDIA Recall:    0.99804688
Fault Recall:   0.99843505
TFLite Size:    40.484375 KB
Inference Time: 0.003020 ms/sample


In [12]:
path = TFLITE_DIR / "MLP_Original_quant.tflite"
interpreter = tf.lite.Interpreter(model_path=str(path))
interpreter.allocate_tensors()

for op in interpreter._get_ops_details():
    print(op['op_name'])

QUANTIZE
SHAPE
STRIDED_SLICE
PACK
RESHAPE
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
LOGISTIC
QUANTIZE
DELEGATE


c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [13]:
path = TFLITE_DIR / "MLP_Original_baseline.tflite"
interpreter = tf.lite.Interpreter(model_path=str(path))
interpreter.allocate_tensors()

for op in interpreter._get_ops_details():
    print(op['op_name'])

SHAPE
STRIDED_SLICE
PACK
RESHAPE
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
LOGISTIC
DELEGATE


# batch check

In [44]:
model_path = TFLITE_DIR / "CNN1D_Node_baseline.tflite"

interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=8)
input_details = interpreter.get_input_details()[0]

print("Shape:", input_details["shape"])
print("Shape signature:", input_details["shape_signature"])

Shape: [ 1 83  6]
Shape signature: [-1 83  6]


# Threads test check

In [9]:
NUM_THREADS = 8

def test_tflite_threads(model_path, X_timing):
    for threads in [1, 2, 4, 8]:
        interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=threads)

        input_details = interpreter.get_input_details()[0]
        input_index = input_details["index"]

        interpreter.resize_tensor_input(input_index, [len(X_timing), *X_timing.shape[1:]], strict=False)
        interpreter.allocate_tensors()

        X_ready = X_timing.astype(np.float32)

        interpreter.set_tensor(input_index, X_ready)
        interpreter.invoke()

        times = []

        for _ in range(20):
            start = timeit.default_timer()
            interpreter.invoke()
            end = timeit.default_timer()

            times.append(((end - start) * 1000) / len(X_timing))

        print(f"Threads {threads}: {np.mean(times):.6f} ms/sample")

In [10]:
model_path = TFLITE_DIR / "MLP_Original_quant.tflite"

X_timing = X_test_temporal[timing_idx]

test_tflite_threads(model_path, X_timing)

ValueError: Cannot set tensor: Got value of type FLOAT32 but expected type UINT8 for input 0, name: serving_default_args_0:0 

In [20]:
import pathlib
import numpy as np
import tensorflow as tf
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

KERAS_DIR = pathlib.Path("../Models/keras")
TFLITE_DIR = pathlib.Path("../Models/tflite")

models = {
    "Original": "LSTM_HPO.keras",
    "Node": "LSTM_NodePruned.keras",
    "Weight": "LSTM_WeightPruned.keras",
    "WeightNode": "LSTM_WeightNodePruned.keras",
    "NodeWeight": "LSTM_NodeWeightPruned.keras"
}

data = np.load("../data/fdia_dataset_processed.npz")
X_train = data["X_train"].transpose(0, 2, 1).astype(np.float32)
X_test = data["X_test"].transpose(0, 2, 1).astype(np.float32)
y_test = data["y_test"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


def rebuild_with_unroll(model, input_shape):
    """Rebuilds the model with unroll=True on every LSTM layer, so the
    TFLite converter can produce a fixed, fused graph instead of a
    dynamic while_loop. Weights are copied over unchanged."""
    new_model = tf.keras.Sequential()
    new_model.add(tf.keras.layers.Input(shape=input_shape))

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.LSTM):
            new_model.add(tf.keras.layers.LSTM(
                layer.units,
                return_sequences=layer.return_sequences,
                unroll=True
            ))
        elif isinstance(layer, tf.keras.layers.Dense):
            new_model.add(tf.keras.layers.Dense(layer.units, activation=layer.activation))

    new_model.set_weights(model.get_weights())
    new_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Sanity check: predictions must match before/after rebuild
    diff = np.max(np.abs(model.predict(X_test[:5], verbose=0) - new_model.predict(X_test[:5], verbose=0)))
    print(f"  Rebuild diff check: {diff:.2e}")

    return new_model


def build_converter(model):
    input_shape = model.inputs[0].shape[1:]
    concrete_func = tf.function(model).get_concrete_function(
        tf.TensorSpec([None, *input_shape], model.inputs[0].dtype)
    )
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    return tf.lite.TFLiteConverter.from_concrete_functions([frozen_func], model)


for name, filename in models.items():
    print(f"\nConverting: {name}")
    model = tf.keras.models.load_model(KERAS_DIR / filename, compile=False)
    model = rebuild_with_unroll(model, input_shape=(83, 6))

    # Float TFLite
    converter = build_converter(model)
    tflite_model = converter.convert()
    baseline_path = TFLITE_DIR / f"LSTM_{name}_baseline.tflite"
    baseline_path.write_bytes(tflite_model)

    # Dynamic range quantization (not full int8 — LSTM-specific)
    converter = build_converter(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model_quant = converter.convert()
    quant_path = TFLITE_DIR / f"LSTM_{name}_quant.tflite"
    quant_path.write_bytes(tflite_model_quant)

    print(f"Baseline: {baseline_path.stat().st_size / 1024:.2f} KB")
    print(f"Quantized: {quant_path.stat().st_size / 1024:.2f} KB")

for name in models:
    path = TFLITE_DIR / f"LSTM_{name}_quant.tflite"
    interpreter = tf.lite.Interpreter(model_path=str(path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    print(f"\n{name}")
    print("Input:", input_details["dtype"])
    print("Shape signature:", input_details["shape_signature"])

X_train: (38880, 83, 6)
X_test: (9720, 83, 6)

Converting: Original


  Rebuild diff check: 0.00e+00
INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpgclvmx4z\assets


INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpgclvmx4z\assets


INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpr9zqu_hw\assets


INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpr9zqu_hw\assets


...

INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpzpa6nxiy\assets


INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpzpa6nxiy\assets


Baseline: 1213.13 KB
Quantized: 852.30 KB

Original
Input: <class 'numpy.float32'>
Shape signature: [-1 83  6]


c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Node
Input: <class 'numpy.float32'>
Shape signature: [-1 83  6]

Weight
Input: <class 'numpy.float32'>
Shape signature: [-1 83  6]

WeightNode
Input: <class 'numpy.float32'>
Shape signature: [-1 83  6]

NodeWeight
Input: <class 'numpy.float32'>
Shape signature: [-1 83  6]


# LSTM Tests

In [21]:
## LSTM Models tflite
lstm_tflite_models = [
    ("LSTM_Original_baseline", "LSTM_Original_baseline.tflite", "LSTM_HPO.keras"),
    ("LSTM_Original_quant", "LSTM_Original_quant.tflite", "LSTM_HPO.keras"),

    ("LSTM_Node_baseline", "LSTM_Node_baseline.tflite", "LSTM_NodePruned.keras"),
    ("LSTM_Node_quant", "LSTM_Node_quant.tflite", "LSTM_NodePruned.keras"),

    ("LSTM_Weight_baseline", "LSTM_Weight_baseline.tflite", "LSTM_WeightPruned.keras"),
    ("LSTM_Weight_quant", "LSTM_Weight_quant.tflite", "LSTM_WeightPruned.keras"),

    ("LSTM_WeightNode_baseline", "LSTM_WeightNode_baseline.tflite", "LSTM_WeightNodePruned.keras"),
    ("LSTM_WeightNode_quant", "LSTM_WeightNode_quant.tflite", "LSTM_WeightNodePruned.keras"),

    ("LSTM_NodeWeight_baseline", "LSTM_NodeWeight_baseline.tflite", "LSTM_NodeWeightPruned.keras"),
    ("LSTM_NodeWeight_quant", "LSTM_NodeWeight_quant.tflite", "LSTM_NodeWeightPruned.keras")
]

In [10]:
## Run tests LSTM tflite
for name, tflite_file, keras_file in lstm_tflite_models:
    result = evaluate_tflite(
        TFLITE_DIR / tflite_file,
        KERAS_DIR / keras_file,
        name,
        "LSTM"
    )

    results.append(result)

NameError: name 'lstm_tflite_models' is not defined

In [11]:
## Final results table
final_df = pd.DataFrame(results)

final_df["accuracy"] = final_df["accuracy"].round(6)
final_df["fdia_recall"] = final_df["fdia_recall"].round(6)
final_df["fault_recall"] = final_df["fault_recall"].round(6)
final_df["model_size_kb"] = final_df["model_size_kb"].round(3)
final_df["inference_time_ms"] = final_df["inference_time_ms"].round(6)

final_df.to_csv("all_models_tflite_comparison.csv", index=False)

print(f"Saved: all_models_tflite_comparison.csv ({len(final_df)} rows)")
final_df

Saved: all_models_tflite_comparison.csv (20 rows)


,architecture,model,parameters,accuracy,fdia_recall,fault_recall,model_size_kb,inference_time_ms
0,CNN1D,CNN1D_Original_baseline,92289,0.998354,0.999349,0.997457,367.723,0.037456
1,CNN1D,CNN1D_Original_quant,92289,0.997531,0.999349,0.995892,105.391,0.025078
2,CNN1D,CNN1D_Node_baseline,46596,0.991667,0.999566,0.984546,189.352,0.022876
3,CNN1D,CNN1D_Node_quant,46596,0.988992,0.999783,0.979264,58.664,0.014801
4,CNN1D,CNN1D_Weight_baseline,92289,0.999074,0.999349,0.998826,367.609,0.040831
5,CNN1D,CNN1D_Weight_quant,92289,0.998663,0.999566,0.997848,105.281,0.024460
6,CNN1D,CNN1D_WeightNode_baseline,48751,0.992181,0.999566,0.985524,197.730,0.022103
7,CNN1D,CNN1D_WeightNode_quant,48751,0.989506,0.999566,0.980438,60.805,0.014871
8,CNN1D,CNN1D_NodeWeight_baseline,46596,0.998765,0.999349,0.998239,189.328,0.021948
9,CNN1D,CNN1D_NodeWeight_quant,46596,0.997840,0.998698,0.997066,58.648,0.014835


In [20]:
def test_threads(model_path, X_timing):
    for threads in [1, 2, 4, 8]:
        interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=threads)
        input_details = interpreter.get_input_details()[0]
        input_index = input_details["index"]

        interpreter.resize_tensor_input(input_index, [len(X_timing), *X_timing.shape[1:]], strict=False)
        interpreter.allocate_tensors()

        input_details = interpreter.get_input_details()[0]
        X_ready = X_timing.astype(np.float32)

        if input_details["dtype"] != np.float32:
            scale, zero_point = input_details["quantization"]
            limits = np.iinfo(input_details["dtype"])
            X_ready = np.clip(np.round(X_ready / scale + zero_point), limits.min, limits.max).astype(input_details["dtype"])

        interpreter.set_tensor(input_index, X_ready)
        interpreter.invoke()

        times = []

        for _ in range(20):
            start = timeit.default_timer()
            interpreter.invoke()
            end = timeit.default_timer()
            times.append(((end - start) * 1000) / len(X_timing))

        print(f"Threads {threads}: {np.mean(times):.6f} ms/sample")


X_timing = X_test[timing_idx]

print("FP32")
test_threads(TFLITE_DIR / "MLP_Original_baseline.tflite", X_timing)

print("\nQuant")
test_threads(TFLITE_DIR / "MLP_Original_quant.tflite", X_timing)

FP32
Threads 1: 0.002064 ms/sample
Threads 2: 0.001178 ms/sample
Threads 4: 0.000891 ms/sample
Threads 8: 0.000995 ms/sample

Quant
Threads 1: 0.004273 ms/sample
Threads 2: 0.003439 ms/sample
Threads 4: 0.003161 ms/sample
Threads 8: 0.003229 ms/sample


In [21]:
for file in ["MLP_Original_baseline.tflite", "MLP_Original_quant.tflite"]:
    print(f"\n{file}")

    interpreter = tf.lite.Interpreter(
        model_path=str(TFLITE_DIR / file),
        num_threads=4
    )
    interpreter.allocate_tensors()

    print("Input:", interpreter.get_input_details()[0]["dtype"])
    print("Output:", interpreter.get_output_details()[0]["dtype"])

    for op in interpreter._get_ops_details():
        print(op["op_name"])


MLP_Original_baseline.tflite
Input: <class 'numpy.float32'>
Output: <class 'numpy.float32'>
SHAPE
STRIDED_SLICE
PACK
RESHAPE
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
LOGISTIC
DELEGATE

MLP_Original_quant.tflite
Input: <class 'numpy.uint8'>
Output: <class 'numpy.uint8'>
QUANTIZE
SHAPE
STRIDED_SLICE
PACK
RESHAPE
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
FULLY_CONNECTED
LOGISTIC
QUANTIZE
DELEGATE


In [25]:
def representative_data_gen():
    for sample in X_test[:100]:
        yield [sample[np.newaxis, ...]]

# Reconverte SÓ o MLP_Original com o método ANTIGO (que funcionava)
model = tf.keras.models.load_model(KERAS_DIR / "MLP_HPO.keras", compile=False)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
(TFLITE_DIR / "MLP_Original_baseline_OLD.tflite").write_bytes(tflite_model)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_model_quant = converter.convert()
(TFLITE_DIR / "MLP_Original_quant_OLD.tflite").write_bytes(tflite_model_quant)

# Mede os dois, compara com o resultado que você já tem

INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpz7j2gxjq\assets


INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpz7j2gxjq\assets


Saved artifact at 'C:\Users\lcdlu\AppData\Local\Temp\tmpz7j2gxjq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6, 83), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2289341943248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341945552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341945360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341945936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341945744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341947088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946

INFO:tensorflow:Assets written to: C:\Users\lcdlu\AppData\Local\Temp\tmpyzdcw3z6\assets


Saved artifact at 'C:\Users\lcdlu\AppData\Local\Temp\tmpyzdcw3z6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6, 83), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2289341943248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341945552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341945360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341945936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341945744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341947088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2289341946

c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


59520

In [26]:
## Compare OLD (from_keras_model) vs NEW (from_concrete_functions) — MLP_Original_quant
old_path = TFLITE_DIR / "MLP_Original_quant_OLD.tflite"
new_path = TFLITE_DIR / "MLP_Original_quant.tflite"

X_timing_mlp = X_test[timing_idx]

old_time = measure_inference_time_tflite(old_path, X_timing_mlp, RUNS)
new_time = measure_inference_time_tflite(new_path, X_timing_mlp, RUNS)

print(f"OLD (from_keras_model):       {old_time:.6f} ms/sample")
print(f"NEW (from_concrete_functions): {new_time:.6f} ms/sample")
print(f"Ratio (NEW/OLD): {new_time/old_time:.2f}x")

# Também confere se ainda dá o mesmo resultado de acurácia, pra garantir
# que a comparação de tempo não está sendo distorcida por modelos diferentes
old_preds = predict_tflite(old_path, X_test)
new_preds = predict_tflite(new_path, X_test)

old_acc = accuracy_score(y_test, (old_preds >= 0.5).astype(int))
new_acc = accuracy_score(y_test, (new_preds >= 0.5).astype(int))

print(f"\nOLD accuracy: {old_acc:.6f}")
print(f"NEW accuracy: {new_acc:.6f}")

c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


OLD (from_keras_model):       0.003311 ms/sample
NEW (from_concrete_functions): 0.003069 ms/sample
Ratio (NEW/OLD): 0.93x

OLD accuracy: 0.998148
NEW accuracy: 0.997634


In [16]:
## Quick XNNPack on/off comparison — MLP baseline vs quant
def measure_inference_time_tflite_no_xnnpack(model_path, X_timing, runs=100):
    interpreter = tf.lite.Interpreter(
        model_path=str(model_path),
        num_threads=8,
        experimental_delegates=[]
    )

    input_details = interpreter.get_input_details()[0]
    input_index = input_details["index"]

    interpreter.resize_tensor_input(input_index, [len(X_timing), *X_timing.shape[1:]], strict=False)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    X_ready = X_timing.astype(np.float32)

    if input_details["dtype"] != np.float32:
        scale, zero_point = input_details["quantization"]
        limits = np.iinfo(input_details["dtype"])
        X_ready = np.clip(np.round(X_ready / scale + zero_point), limits.min, limits.max).astype(input_details["dtype"])

    interpreter.set_tensor(input_index, X_ready)
    interpreter.invoke()

    run_times = []
    for _ in range(runs):
        start = timeit.default_timer()
        interpreter.invoke()
        end = timeit.default_timer()
        run_times.append(((end - start) * 1000) / len(X_timing))

    return np.mean(run_times)


X_timing_mlp = X_test[timing_idx]

baseline_time = measure_inference_time_tflite_no_xnnpack(
    TFLITE_DIR / "MLP_Original_baseline.tflite", X_timing_mlp, RUNS
)
quant_time = measure_inference_time_tflite_no_xnnpack(
    TFLITE_DIR / "MLP_Original_quant.tflite", X_timing_mlp, RUNS
)

print(f"Baseline (no XNNPack): {baseline_time:.6f} ms/sample")
print(f"Quant (no XNNPack):    {quant_time:.6f} ms/sample")
print(f"Quant is {'faster' if quant_time < baseline_time else 'slower'} than baseline")

Baseline (no XNNPack): 0.000393 ms/sample
Quant (no XNNPack):    0.001320 ms/sample
Quant is slower than baseline


In [36]:
models = {
    "Baseline": TFLITE_DIR / "MLP_Original_baseline.tflite",
    "Quantized": TFLITE_DIR / "MLP_Original_quant.tflite"
}

X_timing = X_test[timing_idx]
RUNS = 100
NUM_THREADS = 8

for name, model_path in models.items():
    interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=NUM_THREADS)

    input_details = interpreter.get_input_details()[0]
    input_index = input_details["index"]

    interpreter.resize_tensor_input(input_index, [len(X_timing), *X_timing.shape[1:]], strict=False)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    X_ready = X_timing.astype(np.float32)

    if input_details["dtype"] != np.float32:
        scale, zero_point = input_details["quantization"]
        limits = np.iinfo(input_details["dtype"])
        X_ready = np.clip(np.round(X_ready / scale + zero_point), limits.min, limits.max).astype(input_details["dtype"])

    interpreter.set_tensor(input_index, X_ready)

    # Warm-up
    interpreter.invoke()

    times = []

    for _ in range(RUNS):
        start = timeit.default_timer()
        interpreter.invoke()
        end = timeit.default_timer()

        times.append(((end - start) * 1000) / len(X_timing))

    print(f"{name}: {np.mean(times):.6f} ms/sample")

Baseline: 0.000460 ms/sample
Quantized: 0.001389 ms/sample
